In [0]:
# 1. The "Ingestion & Processing" Layer (PySpark)
# In Databricks, Spark is native. This code simulates receiving "dirty" sensor data from a stream (Kafka) and cleaning it for storage in a Data Lake (Delta Lake/S3).

# --- STEP 1: Spark Data Cleaning (The 'Compute' Phase) ---
from pyspark.sql.functions import col, current_timestamp, avg

# Simulating incoming sensor data (Pressure from Water Network)
data = [("Sensor_A1", 45.2), ("Sensor_A2", 38.5), ("Sensor_A1", 44.8), ("Sensor_B1", 12.0)]
columns = ["sensor_id", "pressure_psi"]

df = spark.createDataFrame(data, columns)
display(df)

In [0]:
# Logic: Identify low pressure (Potential Leak < 40 PSI)
cleaned_df = df.withColumn("ingestion_time", current_timestamp()) \
               .filter(col("pressure_psi") > 0) # Remove noise

# Aggregating to find average pressure by sector
stats_df = cleaned_df.groupBy("sensor_id").agg(avg("pressure_psi").alias("avg_pressure"))

stats_df.show()
# This results would typically be saved to Snowflake or a Delta Table here.

In [0]:
# 2. The "Advanced Intelligence" Layer (Ray)
# While Spark handles the table above, we use Ray to run a parallel simulation or a complex model that doesn't fit into a standard SQL-like table. Note: In Databricks, you can install Ray via %pip install ray.

# --- STEP 2: Ray Distributed Task (The 'Analyze' Phase) ---
!pip install ray
import ray

In [0]:
# Initialize Ray (In Databricks, this connects to the cluster nodes)
if not ray.is_initialized():
    ray.init()

@ray.remote
def predict_leak_risk(sensor_id, avg_pressure):
    """
    Complex logic that might use a pre-trained TensorFlow 
    or Scikit-learn model to predict the probability of a burst.
    """
    import time
    time.sleep(1) # Simulating heavy computation
    risk_score = 1.0 if avg_pressure < 35 else 0.1
    return {sensor_id: risk_score}

# Execute the function in parallel for all sensors found by Spark
sensor_list = stats_df.collect()
futures = [predict_leak_risk.remote(row['sensor_id'], row['avg_pressure']) for row in sensor_list]

# Get results back
final_risks = ray.get(futures)
print(f"Real-time Risk Assessment: {final_risks}")

In [0]:
# Handling the GCS connection carefully for Community Edition
import pandas as pd

if ray.is_initialized():
    ray.shutdown()

# Initialize Ray locally on the driver node
ray.init(ignore_reinit_error=True)

@ray.remote
def analyze_leak_risk(sensor_id, pressure):
    """
    Complex Distributed Task: 
    In a real scenario, this would load a Scikit-learn or TensorFlow model.
    """
    import time
    time.sleep(0.5) # Simulate complex model inference
    
    # Logic: Risk increases as pressure drops below 35 PSI
    risk_level = "HIGH" if pressure < 35 else "LOW"
    return {"id": sensor_id, "pressure": round(pressure, 2), "risk": risk_level}

# Convert Spark DataFrame to pandas for iteration
results_df = stats_df.toPandas()

# Launching Ray Tasks in Parallel
print("\n--- Launching Ray Distributed Tasks ---")
futures = [analyze_leak_risk.remote(row['sensor_id'], row['avg_pressure']) for _, row in results_df.iterrows()]

# Collect results
results = ray.get(futures)

# --- FINAL OUTPUT ---
results_df = pd.DataFrame(results)
print("\n--- Final Risk Intelligence Table ---")
print(results_df)

# Shutdown Ray to free up memory for the next run
ray.shutdown()

In [0]:
from pyspark.sql.functions import udf, col, avg
from pyspark.sql.types import StringType
import time

# 2. THE CUSTOM LOGIC (The "Brain")
# This is where your AI model or complex math would live
def analyze_risk_logic(pressure):
    """
    Simulates a complex Python model.
    Spark will run this in parallel on all worker nodes.
    """
    if pressure is None: return "UNKNOWN"
    # Imagine calling a scikit-learn .predict() here
    if pressure < 30:
        return "CRITICAL LEAK"
    elif pressure < 40:
        return "WARNING"
    else:
        return "STABLE"

# Register the Python function as a Spark UDF
risk_udf = udf(analyze_risk_logic, StringType())

# 3. THE EXECUTION
# We aggregate the data and THEN apply the intelligence
final_results = df.groupBy("sensor_id") \
                  .agg(avg("pressure_psi").alias("avg_p")) \
                  .withColumn("risk_status", risk_udf(col("avg_p")))

# Display results
print("--- Integrated Spark Risk Intelligence ---")
final_results.show()

# 4. EXPORT TO SQL (For Dashboards)
final_results.createOrReplaceTempView("v_leak_intelligence")

# Bank Example
## Credit Risk

In [0]:
# 1. Ingestion: Upload the credit_risk_dataset.csv to a Unity Catalog Volume.

# * Bronze (Raw): Read the CSV exactly as it is. This is your "Source of Truth."
# * Silver (Cleaned): This is where you handle the specific issues in this Kaggle set (imputing missing interest rates, removing outliers in person age, etc.).
# * Gold (Risk Features): Creating the final "Default Probability" flags for the business.

# 2. Implementation Code (Spark-Native)

# First Step:  
## GO to Create Medallion (schemas created)
$$bronze \rightarrow \rightarrow silver \rightarrow gold$$

In [0]:
# Step A: Bronze (The Raw Table)
# We read from the Volume and save it as a permanent table in the Bronze Schema.
# Path to your uploaded Kaggle file in the Volume
csv_path = "/Volumes/workspace/bronze/landing_zone/credit_risk_dataset.csv"

# Read Raw
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(csv_path)

# Save to Bronze Schema
df_raw.write.mode("overwrite").saveAsTable("workspace.bronze.credit_risk_raw")

In [0]:
display(df_raw)

In [0]:
%sql
select person_age 
from workspace.bronze.credit_risk_raw;

select * from workspace.bronze.credit_risk_raw
where person_age > 50

In [0]:
# Step B: Silver (The Cleaned Table)
# We apply the "Bank Rules" (handling nulls and filtering outliers) and save it to the Silver Schema.

from pyspark.sql.functions import when, col, mean, split

# Read from Bronze

bronze_df = spark.read.table("workspace.bronze.credit_risk_raw")

# Cleaning Logic: Fill nulls & remove age outliers
avg_rate = bronze_df.select(mean("loan_int_rate")).collect()[0][0]

df_silver = bronze_df.withColumn("loan_int_rate", 
    when(col("loan_int_rate").isNull(), avg_rate).otherwise(col("loan_int_rate"))) \
    .filter(col("person_age") < 100)

# Save to Silver
df_silver.write.mode("overwrite").saveAsTable("workspace.silver.credit_risk_cleaned")

In [0]:
# Layer 3: Gold (The "Knowledge")
# We apply the high-level business logic (Credit Risk segments) and save to gold.
silver_df = spark.read.table("workspace.silver.credit_risk_cleaned")

# Business Logic: Segmenting users by risk
df_gold = silver_df.withColumn("risk_segment", 
    when((col("loan_percent_income") > 0.3) & (col("person_home_ownership") == 'RENT'), "HIGH_RISK")
    .otherwise("STANDARD_RISK")
)

# Save to Gold
df_gold.write.mode("overwrite").saveAsTable("workspace.gold.credit_risk_final")

# German Credit Data Pipeline
## Arquitectura Medallion: Bronze → Silver → Gold
### Objetivo: Modelo de Clasificación para Probabilidad de Pago

In [0]:
# PASO 1: Instalación de dependencias y descarga de datos desde UCI
import requests
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import seaborn as sns

# Descargar datos de UCI ML Repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
response = requests.get(url)

# Los datos vienen sin headers, definimos los nombres de columnas según la documentación
column_names = [
    'checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount',
    'savings_status', 'employment', 'installment_commitment', 'personal_status',
    'other_parties', 'residence_since', 'property_magnitude', 'age', 'other_payment_plans',
    'housing', 'existing_credits', 'job', 'num_dependents', 'own_telephone', 'foreign_worker',
    'class'
]

# Crear DataFrame pandas
from io import StringIO
df_pandas = pd.read_csv(StringIO(response.text), sep=' ', names=column_names)

print(f"Datos descargados: {df_pandas.shape[0]} filas, {df_pandas.shape[1]} columnas")
print("\nPrimeras filas:")
display(df_pandas.head())

In [0]:
# PASO 2: Crear estructura medallion en Unity Catalog
# Verificar/crear catálogo y esquemas

try:
    spark.sql("CREATE CATALOG IF NOT EXISTS clase_bigdata")
    print("✓ Catálogo 'clase_bigdata' creado/verificado")
except Exception as e:
    print(f"Catálogo ya existe o error: {e}")

# Crear esquemas bronze, silver, gold
for schema in ['bronze', 'silver', 'gold']:
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS clase_bigdata.{schema}")
        print(f"✓ Schema '{schema}' creado/verificado")
    except Exception as e:
        print(f"Schema {schema} ya existe o error: {e}")

print("\n✓ Arquitectura medallion lista: clase_bigdata.{bronze|silver|gold}")

In [0]:
# PASO 3: BRONZE LAYER - Ingesta de datos raw
# Convertir pandas a Spark DataFrame
df_spark = spark.createDataFrame(df_pandas)

# Agregar metadata de ingesta
df_bronze = df_spark.withColumn("ingestion_timestamp", F.current_timestamp()) \
                    .withColumn("source", F.lit("UCI_ML_Repository"))

# Guardar en tabla bronze
df_bronze.write.mode("overwrite").saveAsTable("clase_bigdata.bronze.german_credit_raw")

print("✓ Datos guardados en: clase_bigdata.bronze.german_credit_raw")
print(f"\nRegistros en Bronze: {df_bronze.count()}")

# Verificar tabla creada
display(spark.sql("SELECT * FROM clase_bigdata.bronze.german_credit_raw LIMIT 5"))

## Análisis Exploratorio de Datos (EDA)
### Entendiendo la estructura y calidad de los datos

In [0]:
# PASO 4: EDA - Información Básica y Estadísticas Descriptivas

# Cargar datos desde Bronze
df_eda = spark.read.table("clase_bigdata.bronze.german_credit_raw")
df_eda_pandas = df_eda.toPandas()

print("="*80)
print("INFORMACIÓN GENERAL DEL DATASET")
print("="*80)
print(f"\nDimensiones: {df_eda_pandas.shape[0]} filas x {df_eda_pandas.shape[1]} columnas")
print(f"\nTamaño en memoria: {df_eda_pandas.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n" + "="*80)
print("TIPOS DE DATOS Y VALORES NULOS")
print("="*80)
info_df = pd.DataFrame({
    'Columna': df_eda_pandas.columns,
    'Tipo': df_eda_pandas.dtypes.values,
    'Nulos': df_eda_pandas.isnull().sum().values,
    '% Nulos': (df_eda_pandas.isnull().sum().values / len(df_eda_pandas) * 100).round(2),
    'Valores Únicos': df_eda_pandas.nunique().values
})
print(info_df.to_string(index=False))

print("\n" + "="*80)
print("ESTADÍSTICAS DESCRIPTIVAS - VARIABLES NUMÉRICAS")
print("="*80)
numeric_cols = df_eda_pandas.select_dtypes(include=[np.number]).columns
print(df_eda_pandas[numeric_cols].describe().round(2))

In [0]:
# PASO 5: EDA - Análisis de Variable Target (class)
# 1 = buen cliente (paga), 2 = mal cliente (no paga/default)

print("="*80)
print("ANÁLISIS DE VARIABLE TARGET: 'class'")
print("="*80)
print("\nDistribución de clases:")
print(df_eda_pandas['class'].value_counts().sort_index())
print(f"\nPorcentajes:")
print(df_eda_pandas['class'].value_counts(normalize=True).sort_index().apply(lambda x: f"{x*100:.1f}%"))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
class_counts = df_eda_pandas['class'].value_counts().sort_index()
axes[0].bar(['Buen Cliente (1)', 'Mal Cliente (2)'], class_counts.values, color=['green', 'red'], alpha=0.7)
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Clases', fontsize=14, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Gráfico de pastel
axes[1].pie(class_counts.values, labels=['Buen Cliente (70%)', 'Mal Cliente (30%)'], 
            autopct='%1.1f%%', colors=['green', 'red'], startangle=90)
axes[1].set_title('Proporción de Clases', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n⚠️ Observación: Dataset ligeramente desbalanceado (70%-30%)")

In [0]:
# PASO 6: EDA - Análisis de Variables Numéricas

print("="*80)
print("ANÁLISIS DE VARIABLES NUMÉRICAS")
print("="*80)

# Variables numéricas clave
numeric_features = ['duration', 'credit_amount', 'age', 'installment_commitment', 
                    'residence_since', 'existing_credits', 'num_dependents']

# Distribución de variables numéricas
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for idx, col in enumerate(numeric_features):
    if idx < len(axes):
        axes[idx].hist(df_eda_pandas[col], bins=30, color='steelblue', alpha=0.7, edgecolor='black')
        axes[idx].set_title(f'{col}', fontsize=11, fontweight='bold')
        axes[idx].set_xlabel('Valor')
        axes[idx].set_ylabel('Frecuencia')
        axes[idx].grid(axis='y', alpha=0.3)
        
        # Estadísticas en el gráfico
        mean_val = df_eda_pandas[col].mean()
        median_val = df_eda_pandas[col].median()
        axes[idx].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Media: {mean_val:.1f}')
        axes[idx].axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Mediana: {median_val:.1f}')
        axes[idx].legend(fontsize=8)

# Remover ejes extras
for idx in range(len(numeric_features), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.show()

print("\n✓ Análisis de distribuciones completado")

In [0]:
# PASO 7: EDA - Correlación entre Variables Numéricas y Target

print("="*80)
print("ANÁLISIS DE CORRELACIÓN")
print("="*80)

# Matriz de correlación
corr_matrix = df_eda_pandas[numeric_features + ['class']].corr()

# Visualizar matriz de correlación
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title('Matriz de Correlación - Variables Numéricas', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Correlación con la variable target
print("\nCorrelación con variable TARGET ('class'):")
print("="*50)
target_corr = corr_matrix['class'].drop('class').sort_values(ascending=False)
for feature, corr_val in target_corr.items():
    print(f"{feature:30s}: {corr_val:+.4f}")

print("\n⚠️ Interpretación:")
print("   - Correlaciones positivas: a mayor valor, más probable ser 'mal cliente'")
print("   - Correlaciones negativas: a mayor valor, más probable ser 'buen cliente'")

In [0]:
# PASO 8: EDA - Relación Features Numéricas vs Target

print("="*80)
print("RELACIÓN ENTRE FEATURES NUMÉRICAS Y TARGET")
print("="*80)

# Boxplots para comparar distribuciones por clase
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

key_features = ['credit_amount', 'duration', 'age', 'installment_commitment', 'existing_credits']

for idx, col in enumerate(key_features):
    if idx < len(axes):
        df_eda_pandas.boxplot(column=col, by='class', ax=axes[idx])
        axes[idx].set_title(f'{col} por Clase')
        axes[idx].set_xlabel('Clase (1=Bueno, 2=Malo)')
        axes[idx].set_ylabel(col)
        axes[idx].get_figure().suptitle('')  # Remover título automático

# Remover eje extra
if len(key_features) < len(axes):
    fig.delaxes(axes[-1])

plt.suptitle('Distribución de Features por Clase de Cliente', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("\n✓ Visualización de relaciones completada")

## SILVER LAYER - Transformación y Limpieza
### Codificación de variables categóricas y normalización

In [0]:
# PASO 9: SILVER LAYER - Decodificar variables categóricas y normalizar
# Las variables categóricas vienen codificadas (A11, A12, etc.)
# Vamos a transformarlas a valores interpretables

from pyspark.sql.functions import when, col

# Leer desde Bronze
df_bronze = spark.read.table("clase_bigdata.bronze.german_credit_raw")

# Mapeo de variables categóricas (basado en documentación UCI)
checking_status_map = {
    'A11': '<0 DM', 'A12': '0-200 DM', 'A13': '>200 DM', 'A14': 'no checking account'
}

credit_history_map = {
    'A30': 'no credits taken', 'A31': 'all paid', 'A32': 'existing paid',
    'A33': 'delay in past', 'A34': 'critical account'
}

purpose_map = {
    'A40': 'car (new)', 'A41': 'car (used)', 'A42': 'furniture/equipment',
    'A43': 'radio/tv', 'A44': 'domestic appliances', 'A45': 'repairs',
    'A46': 'education', 'A47': '(vacation)', 'A48': 'retraining',
    'A49': 'business', 'A410': 'others'
}

savings_map = {
    'A61': '<100 DM', 'A62': '100-500 DM', 'A63': '500-1000 DM',
    'A64': '>1000 DM', 'A65': 'unknown/no savings'
}

employment_map = {
    'A71': 'unemployed', 'A72': '<1 year', 'A73': '1-4 years',
    'A74': '4-7 years', 'A75': '>7 years'
}

# Aplicar mapeos
df_silver = df_bronze

for old_val, new_val in checking_status_map.items():
    df_silver = df_silver.withColumn('checking_status', 
        when(col('checking_status') == old_val, new_val).otherwise(col('checking_status')))

for old_val, new_val in credit_history_map.items():
    df_silver = df_silver.withColumn('credit_history', 
        when(col('credit_history') == old_val, new_val).otherwise(col('credit_history')))

for old_val, new_val in purpose_map.items():
    df_silver = df_silver.withColumn('purpose', 
        when(col('purpose') == old_val, new_val).otherwise(col('purpose')))

for old_val, new_val in savings_map.items():
    df_silver = df_silver.withColumn('savings_status', 
        when(col('savings_status') == old_val, new_val).otherwise(col('savings_status')))

for old_val, new_val in employment_map.items():
    df_silver = df_silver.withColumn('employment', 
        when(col('employment') == old_val, new_val).otherwise(col('employment')))

# Transformar variable target: 1 (bueno) -> 0, 2 (malo) -> 1 
# (para que 1 represente DEFAULT/riesgo)
df_silver = df_silver.withColumn('target', 
    when(col('class') == 2, 1).otherwise(0))

# Guardar en Silver
df_silver.write.mode("overwrite").saveAsTable("clase_bigdata.silver.german_credit_transformed")

print("✓ Datos transformados guardados en: clase_bigdata.silver.german_credit_transformed")
print(f"\nRegistros en Silver: {df_silver.count()}")

# Mostrar sample
display(spark.sql("""
    SELECT checking_status, credit_history, purpose, savings_status, 
           employment, credit_amount, duration, age, target
    FROM clase_bigdata.silver.german_credit_transformed 
    LIMIT 5
"""))

## GOLD LAYER - Feature Engineering
### Preparación de features para Machine Learning

In [0]:
# PASO 10: GOLD LAYER - Feature Engineering
# Crear features adicionales y preparar para ML

from pyspark.sql.functions import col, when, round as spark_round

# Leer desde Silver
df_silver = spark.read.table("clase_bigdata.silver.german_credit_transformed")

# Feature Engineering
df_gold = df_silver.withColumn(
    'credit_to_income_ratio', 
    spark_round(col('credit_amount') / (col('duration') + 1), 2)
).withColumn(
    'age_group',
    when(col('age') < 25, 'young')
    .when((col('age') >= 25) & (col('age') < 40), 'adult')
    .when((col('age') >= 40) & (col('age') < 60), 'middle_aged')
    .otherwise('senior')
).withColumn(
    'duration_category',
    when(col('duration') <= 12, 'short')
    .when((col('duration') > 12) & (col('duration') <= 24), 'medium')
    .otherwise('long')
).withColumn(
    'credit_risk_score',
    spark_round(
        (col('duration') * 0.3) + 
        (col('credit_amount') / 1000 * 0.3) + 
        (col('installment_commitment') * 0.2) - 
        (col('age') * 0.2),
        2
    )
).withColumn(
    'high_credit_amount',
    when(col('credit_amount') > 5000, 1).otherwise(0)
).withColumn(
    'long_duration',
    when(col('duration') > 24, 1).otherwise(0)
)

# Seleccionar columnas relevantes para el modelo
feature_columns = [
    'duration', 'credit_amount', 'installment_commitment', 'residence_since',
    'age', 'existing_credits', 'num_dependents', 'credit_to_income_ratio',
    'credit_risk_score', 'high_credit_amount', 'long_duration',
    'checking_status', 'credit_history', 'purpose', 'savings_status', 'employment',
    'target'
]

df_gold = df_gold.select(*feature_columns)

# Guardar en Gold
df_gold.write.mode("overwrite").saveAsTable("clase_bigdata.gold.german_credit_features")

print("✓ Features de ML guardados en: clase_bigdata.gold.german_credit_features")
print(f"\nRegistros en Gold: {df_gold.count()}")
print(f"Features numéricas: 11")
print(f"Features categóricas: 5")
print(f"\nColumnas finales: {len(feature_columns)}")

# Mostrar sample
display(df_gold.limit(5))

## MODELO DE CLASIFICACIÓN
### Random Forest para predicción de probabilidad de pago

In [0]:
# PASO 11: Instalación de librerías necesarias para ML
%pip install category-encoders imbalanced-learn scikit-learn mlflow --quiet
dbutils.library.restartPython()

In [0]:
# PASO 12: Preparación de datos para ML
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from category_encoders import TargetEncoder
import warnings
warnings.filterwarnings('ignore')

# Cargar datos desde Gold
df_gold = spark.read.table("clase_bigdata.gold.german_credit_features").toPandas()

print("="*80)
print("PREPARACIÓN DE DATOS PARA MACHINE LEARNING")
print("="*80)

# Separar features y target
X = df_gold.drop('target', axis=1)
y = df_gold['target']

print(f"\nDimensiones del dataset:")
print(f"  - Features (X): {X.shape}")
print(f"  - Target (y): {y.shape}")
print(f"  - Clases en target: {y.value_counts().to_dict()}")

# Identificar columnas numéricas y categóricas
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"\nFeatures numéricas ({len(numeric_features)}): {numeric_features}")
print(f"\nFeatures categóricas ({len(categorical_features)}): {categorical_features}")

# Split train/test (80-20) con estratificación por target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n\u2713 División train/test completada:")
print(f"  - Train: {X_train.shape[0]} muestras ({y_train.mean()*100:.1f}% positivos)")
print(f"  - Test: {X_test.shape[0]} muestras ({y_test.mean()*100:.1f}% positivos)")

In [0]:
# PASO 13: Construcción del Pipeline de ML y Entrenamiento
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import OneHotEncoder
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import matplotlib.pyplot as plt
import seaborn as sns

print("="*80)
print("ENTRENAMIENTO DEL MODELO")
print("="*80)

# Crear preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features)
    ],
    remainder='drop'
)

# Crear pipeline completo (preprocesamiento + modelo)
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=20,
        min_samples_leaf=10,
        class_weight='balanced',  # Manejo de desbalance
        random_state=42,
        n_jobs=-1
    ))
])

# MLflow experiment tracking
mlflow.set_experiment("/Users/luis.rivera8383@unaula.edu.co/german-credit-classification")

with mlflow.start_run(run_name="RandomForest_Balanced") as run:
    print("\n⏳ Entrenando modelo Random Forest...")
    
    # Entrenar modelo
    model.fit(X_train, y_train)
    
    # Predicciones
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    y_pred_proba_train = model.predict_proba(X_train)[:, 1]
    y_pred_proba_test = model.predict_proba(X_test)[:, 1]
    
    # Métricas
    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)
    roc_auc_train = roc_auc_score(y_train, y_pred_proba_train)
    roc_auc_test = roc_auc_score(y_test, y_pred_proba_test)
    
    print("\n✓ Entrenamiento completado!")
    print(f"\n  - Accuracy Train: {train_score:.4f}")
    print(f"  - Accuracy Test: {test_score:.4f}")
    print(f"  - ROC-AUC Train: {roc_auc_train:.4f}")
    print(f"  - ROC-AUC Test: {roc_auc_test:.4f}")
    
    # Log parameters y metrics
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("accuracy_train", train_score)
    mlflow.log_metric("accuracy_test", test_score)
    mlflow.log_metric("roc_auc_train", roc_auc_train)
    mlflow.log_metric("roc_auc_test", roc_auc_test)
    
    # Log modelo con signature usando cloudpickle
    signature = infer_signature(X_train, model.predict_proba(X_train))
    model_info = mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        signature=signature,
        input_example=X_train.head(3),
        serialization_format='cloudpickle'
    )
    
    print(f"\n✓ Modelo registrado en MLflow")
    print(f"  - Run ID: {run.info.run_id}")
    print(f"  - Model URI: {model_info.model_uri}")

In [0]:
# PASO 14: Evaluación Detallada del Modelo

print("="*80)
print("EVALUACIÓN DETALLADA DEL MODELO")
print("="*80)

# Reporte de clasificación
print("\nREPORTE DE CLASIFICACIÓN - TEST SET:")
print(classification_report(y_test, y_pred_test, target_names=['Buen Cliente (0)', 'Mal Cliente (1)']))

# Matriz de confusión
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Test set confusion matrix
cm_test = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred: Bueno', 'Pred: Malo'],
            yticklabels=['Real: Bueno', 'Real: Malo'])
axes[0].set_title('Matriz de Confusión - Test Set', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Valor Real')
axes[0].set_xlabel('Valor Predicho')

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_test)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_test:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Curva ROC - Test Set', fontsize=12, fontweight='bold')
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Evaluación completada")

In [0]:
# PASO 15: Feature Importance
from sklearn.inspection import permutation_importance

print("="*80)
print("IMPORTANCIA DE FEATURES")
print("="*80)

# Obtener feature importance del Random Forest
feature_names = numeric_features + categorical_features
rf_classifier = model.named_steps['classifier']
importances = rf_classifier.feature_importances_

# Crear DataFrame de importancias
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print("\nTop 10 Features Más Importantes:")
print(importance_df.head(10).to_string(index=False))

# Visualizar
fig, ax = plt.subplots(figsize=(10, 6))
top_n = 15
importance_df_top = importance_df.head(top_n)
ax.barh(range(top_n), importance_df_top['importance'], color='steelblue')
ax.set_yticks(range(top_n))
ax.set_yticklabels(importance_df_top['feature'])
ax.invert_yaxis()
ax.set_xlabel('Importancia', fontsize=11)
ax.set_title(f'Top {top_n} Features Más Importantes', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✓ Análisis de importancia completado")

In [0]:
# PASO 16: Generar Predicciones de Probabilidad para Todos los Clientes

print("="*80)
print("GENERACIÓN DE PROBABILIDADES DE PAGO PARA TODOS LOS CLIENTES")
print("="*80)

# Cargar dataset completo
df_complete = spark.read.table("clase_bigdata.gold.german_credit_features").toPandas()

# Separar features
X_all = df_complete.drop('target', axis=1)
y_all = df_complete['target']

# Generar probabilidades
print("\n⏳ Generando probabilidades de pago...")
proba_all = model.predict_proba(X_all)
proba_default = proba_all[:, 1]  # Probabilidad de NO pagar (clase 1)
proba_pay = proba_all[:, 0]      # Probabilidad de PAGAR (clase 0)

# Crear DataFrame con resultados
results_df = pd.DataFrame({
    'customer_id': range(1, len(df_complete) + 1),
    'actual_class': y_all,
    'probability_pay': proba_pay,
    'probability_default': proba_default,
    'predicted_class': model.predict(X_all),
    'credit_amount': X_all['credit_amount'],
    'duration': X_all['duration'],
    'age': X_all['age']
})

# Clasificar riesgo
results_df['risk_category'] = pd.cut(
    results_df['probability_default'],
    bins=[0, 0.3, 0.5, 0.7, 1.0],
    labels=['Bajo Riesgo', 'Riesgo Moderado', 'Alto Riesgo', 'Riesgo Crítico']
)

print("\n✓ Probabilidades generadas!")
print(f"\nTotal de clientes procesados: {len(results_df)}")

print("\n" + "="*50)
print("DISTRIBUCIÓN POR CATEGORÍA DE RIESGO")
print("="*50)
risk_dist = results_df['risk_category'].value_counts().sort_index()
for category, count in risk_dist.items():
    pct = (count / len(results_df)) * 100
    print(f"{category:20s}: {count:3d} clientes ({pct:5.1f}%)")

print("\n" + "="*50)
print("EJEMPLOS DE CLIENTES CON MAYOR PROBABILIDAD DE PAGO")
print("="*50)
print(results_df.nlargest(5, 'probability_pay')[[
    'customer_id', 'probability_pay', 'probability_default', 
    'risk_category', 'credit_amount', 'duration', 'age'
]].to_string(index=False))

print("\n" + "="*50)
print("EJEMPLOS DE CLIENTES CON MAYOR RIESGO DE DEFAULT")
print("="*50)
print(results_df.nlargest(5, 'probability_default')[[
    'customer_id', 'probability_pay', 'probability_default', 
    'risk_category', 'credit_amount', 'duration', 'age'
]].to_string(index=False))

In [0]:
# PASO 17: Guardar Resultados Finales en Tabla Gold

print("="*80)
print("GUARDANDO RESULTADOS FINALES")
print("="*80)

# Convertir a Spark DataFrame
results_spark = spark.createDataFrame(results_df)

# Guardar en tabla Gold
results_spark.write.mode("overwrite").saveAsTable(
    "clase_bigdata.gold.german_credit_predictions"
)

print("\n✓ Resultados guardados en: clase_bigdata.gold.german_credit_predictions")
print(f"  - Total registros: {results_spark.count()}")
print(f"  - Columnas: {len(results_spark.columns)}")

print("\n" + "="*80)
print("PIPELINE COMPLETO FINALIZADO")
print("="*80)
print("\n✅ RESUMEN DEL PROYECTO:")
print("\n  1. ✓ Ingesta de datos desde UCI ML Repository")
print("  2. ✓ Arquitectura Medallion implementada:")
print("       - Bronze: Datos raw (clase_bigdata.bronze.german_credit_raw)")
print("       - Silver: Datos transformados (clase_bigdata.silver.german_credit_transformed)")
print("       - Gold: Features ML (clase_bigdata.gold.german_credit_features)")
print("  3. ✓ Análisis Exploratorio de Datos (EDA) completado")
print("  4. ✓ Modelo Random Forest entrenado y evaluado")
print("  5. ✓ Probabilidades de pago generadas para todos los clientes")
print("  6. ✓ Resultados almacenados (clase_bigdata.gold.german_credit_predictions)")
print("\n" + "="*80)

# Visualización final de resultados
display(spark.sql("""
    SELECT customer_id, probability_pay, probability_default, 
           risk_category, credit_amount, duration, age,
           CASE WHEN actual_class = 0 THEN 'Buen Cliente' ELSE 'Mal Cliente' END as actual_label
    FROM clase_bigdata.gold.german_credit_predictions
    ORDER BY probability_default DESC
    LIMIT 10
"""))